# Multi choice questions answer
This document focuses on analyzing and finding the answers to the Multi choice questions in Part 1 of the assignment

## Prepare libraries and load data:

In [97]:
import pandas as pd

df_cus = pd.read_parquet('../dataset/cleaned/customers.parquet')
df_geo = pd.read_parquet('../dataset/cleaned/geography.parquet')
df_item = pd.read_parquet('../dataset/cleaned/order_items.parquet')
df_ord = pd.read_parquet('../dataset/cleaned/orders.parquet')
df_pduct = pd.read_parquet('../dataset/cleaned/products.parquet')
df_pmot = pd.read_parquet('../dataset/cleaned/promotions.parquet')
df_ret = pd.read_parquet('../dataset/cleaned/returns.parquet')
df_ship = pd.read_parquet('../dataset/cleaned/shipments.parquet')
df_web = pd.read_parquet('../dataset/cleaned/web_traffic.parquet')


Because the information about the data has already been described in the assignment, we will limit checking the data's `info()`

## Question 1: 
Among customers with more than one order, what is the approximate median number of days between two consecutive purchases (inter-order gap)? (Calculated from `orders.csv`)

In [98]:
# Extract necessary columns
df_ques1 = df_ord[['order_id', 'customer_id', 'order_date']].copy()
# Inspect data types
df_ques1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   order_id     646945 non-null  string        
 1   customer_id  646945 non-null  string        
 2   order_date   646945 non-null  datetime64[ns]
dtypes: datetime64[ns](1), string(2)
memory usage: 14.8 MB


In [99]:
# Keep only customers with multiple orders and sort them
df_ques1 = df_ques1[df_ques1.duplicated(subset=['customer_id'], keep=False)].sort_values(by=['customer_id', 'order_date', 'order_id'])
# Check order frequency per customer
print(df_ques1['customer_id'].value_counts())

customer_id
139050    107
141899    105
141897    103
141898    100
139138     96
         ... 
99852       2
99858       2
99859       2
99891       2
99990       2
Name: count, Length: 67888, dtype: Int64


In [100]:
# Calculate the time gap between consecutive purchases for each customer
df_ques1['inter_order_gap'] = df_ques1.groupby('customer_id', observed=True)['order_date'].diff().dt.days
# Drop the first purchase of each customer (where the gap is NaN)
df_ques1 = df_ques1[df_ques1.groupby('customer_id', observed=True).cumcount() > 0]
# Check the first 5 rows
display(df_ques1.head(5))

,order_id,customer_id,order_date,inter_order_gap
143252,184922,1,2014-05-31,675.0
238890,308113,1,2015-07-31,426.0
374571,483190,1,2017-04-23,632.0
544446,702081,1,2020-02-24,1037.0
586950,756884,1,2021-04-24,425.0


In [101]:
df_ques1.describe()

,order_date,inter_order_gap
count,556699,556699.000000
mean,2017-04-05 16:38:14.536904192,285.592509
min,2012-07-04 00:00:00,0.000000
25%,2015-03-19 00:00:00,46.000000
50%,2016-12-30 00:00:00,144.000000
75%,2019-01-28 00:00:00,357.000000
max,2022-12-31 00:00:00,3785.000000
std,NaN,389.691558


In the descriptive statistics table, for the `inter_order_gap` column, the median value (50%) = **144** (days)
#### -> Choose C

## Question 2: 
Which product segment in `products.csv` has the highest average gross profit margin, using the formula (`price` - `cogs`)/`price`?

In [102]:
# Extract necessary columns
df_ques2 = df_pduct[['segment', 'price', 'cogs']].copy()
# Inspect data types
df_ques2.info()
# Check the unique values and their frequencies in the segment column
print(df_ques2['segment'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   segment  2412 non-null   category
 1   price    2412 non-null   float64 
 2   cogs     2412 non-null   float64 
dtypes: category(1), float64(2)
memory usage: 40.5 KB
segment
Activewear     598
Everyday       405
Performance    347
Balanced       306
Standard       262
Premium        177
All-weather    169
Trendy         148
Name: count, dtype: int64


In [103]:
# Calculate and store the gross profit margin
df_ques2['gross_profit_margin'] = (df_ques2['price'] - df_ques2['cogs']) / df_ques2['price']
# Calculate the average gross profit margin for each segment
mean_by_segment = df_ques2.groupby('segment', observed=True)['gross_profit_margin'].mean()
# Print result
print(mean_by_segment)
print(f"\nHighest segment: {mean_by_segment.idxmax()} with margin {mean_by_segment.max()}")

segment
Activewear     0.265600
All-weather    0.284176
Balanced       0.258038
Everyday       0.236343
Performance    0.263650
Premium        0.285377
Standard       0.313442
Trendy         0.240758
Name: gross_profit_margin, dtype: float64

Highest segment: Standard with margin 0.31344174843884803


#### -> Choose D

## Question 3: 
Among the return records linked to products in the *Streetwear* category (joining `returns` with `products` on `product_id`), which return reason appears the most?

In [104]:
# Extract necessary columns
df_ques3 = df_ret[['return_id', 'product_id', 'return_reason']].copy()
df_support = df_pduct[['product_id', 'category']].copy()
# Inspect data types
df_ques3.info()
df_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   return_id      39939 non-null  string  
 1   product_id     39939 non-null  string  
 2   return_reason  39939 non-null  category
dtypes: category(1), string(2)
memory usage: 663.4 KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   product_id  2412 non-null   string  
 1   category    2412 non-null   category
dtypes: category(1), string(1)
memory usage: 21.5 KB


In [105]:
# Merge product categories into returns using product_id
df_ques3 = df_ques3.merge(df_support, on='product_id', how='left')
# Filter for Streetwear products
df_ques3 = df_ques3[df_ques3['category'] == 'Streetwear']
# Print result
print(df_ques3['return_reason'].value_counts())
mode_reason = df_ques3['return_reason'].mode()[0]
print("\nThe most common reason for returns is:", mode_reason)

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

The most common reason for returns is: wrong_size


#### -> Choose B

## Question 4: 
In `web_traffic.csv`, which `traffic source` has the lowest average `bounce_rate` across all days that source appears in the `traffic_source` column?

In [106]:
# Extract necessary columns
df_ques4 = df_web[['date','bounce_rate', 'traffic_source']].copy()
# Inspect data types
df_ques4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            3652 non-null   datetime64[ns]
 1   bounce_rate     3652 non-null   float64       
 2   traffic_source  3652 non-null   category      
dtypes: category(1), datetime64[ns](1), float64(1)
memory usage: 61.0 KB


In [107]:
# Calculate the average bounce rate per traffic source
mean_by_traffic_source = df_ques4.groupby('traffic_source', observed=True)['bounce_rate'].mean()
# Print result
print(mean_by_traffic_source)
print(f"\nTraffic source: {mean_by_traffic_source.idxmin()} has the lowest average bounce rate")

traffic_source
direct            0.004511
email_campaign    0.004458
organic_search    0.004504
paid_search       0.004478
referral          0.004499
social_media      0.004476
Name: bounce_rate, dtype: float64

Traffic source: email_campaign has the lowest average bounce rate


#### -> Choose C

## Question 5: 
What is the approximate percentage of rows in `order_items.csv` that have a promotion applied (i.e., `promo_id` is not null)?

In [108]:
df_ques5 = df_item.copy()
# Inspect data types
df_ques5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  string 
 1   product_id       714669 non-null  string 
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         714669 non-null  string 
 6   promo_id_2       714669 non-null  string 
dtypes: float64(2), int64(1), string(4)
memory usage: 38.2 MB


In [109]:
percentage_apply_promotion = (df_ques5['promo_id'] != 'None').sum() / len(df_ques5) * 100
print(f"The percentage of the promotion applied is: {percentage_apply_promotion.round(0)}%")

The percentage of the promotion applied is: 39.0%


#### -> Choose C

## Question 6: 
In `customers.csv`, considering customers with a non-null `age_group`, which age group has the highest average number of orders per customer? (total orders / number of customers in the group)

In [110]:
# Extract necessary columns
df_ques6 = df_ord[['customer_id', 'order_id']].copy()
df_support = df_cus[['customer_id', 'age_group']].copy()
# Inspect data types
df_ques6.info()
df_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   customer_id  646945 non-null  string
 1   order_id     646945 non-null  string
dtypes: string(2)
memory usage: 9.9 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype   
---  ------       --------------   -----   
 0   customer_id  121930 non-null  string  
 1   age_group    121930 non-null  category
dtypes: category(1), string(1)
memory usage: 1.0 MB


In [111]:
# Merge customer age group into the orders dataframe using customer_id
df_ques6 = df_ques6.merge(df_support, on='customer_id', how='left')
# Calculate average orders per customer for each age group
mean_by_age_group = df_ques6.groupby('age_group', observed=True)['order_id'].count() / df_ques6.groupby('age_group', observed=True)['customer_id'].nunique()
# Print result
print(mean_by_age_group)
print(f"Age group: {mean_by_age_group.idxmax()} has the highest average orders")

age_group
18-24    7.068577
25-34    7.112230
35-44    7.206159
45-54    7.220264
55+      7.268731
dtype: float64
Age group: 55+ has the highest average orders


#### -> Choose A

## Question 7: 
Which `region` in `geography.csv` generates the highest total revenue in `sales_train.csv`?

In [112]:
# Extract necessary columns
df_ques7 = df_ord[['order_id', 'order_date', 'zip', 'order_status', 'payment_value']].copy()
df_support = df_geo[['zip', 'region']].copy()
# Inspect data types
df_ques7.info()
df_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   order_id       646945 non-null  string        
 1   order_date     646945 non-null  datetime64[ns]
 2   zip            646945 non-null  string        
 3   order_status   646945 non-null  category      
 4   payment_value  646945 non-null  float64       
dtypes: category(1), datetime64[ns](1), float64(1), string(2)
memory usage: 20.4 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39948 entries, 0 to 39947
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   zip     39948 non-null  string  
 1   region  39948 non-null  category
dtypes: category(1), string(1)
memory usage: 351.4 KB


In [113]:
# Merge geography data into the orders dataframe
df_ques7 = df_ques7.merge(df_support, on='zip', how='left')
# Calculate total revenue per region within the timeframe (04/07/2012 - 31/12/2022)
revenue = df_ques7.groupby('region', observed=True)['payment_value'].sum()
# Print result
print(revenue)
print(f"\nRegion: {revenue.idxmax()} has the highest revenue")

region
Central    4.719491e+09
East       7.291151e+09
West       3.670227e+09
Name: payment_value, dtype: float64

Region: East has the highest revenue


#### -> Choose C

## Question 8: 
Among the orders with `order_status` = *cancelled* in `orders.csv`, which payment method is used the most?

In [114]:
# Extract necessary columns
df_ques8 = df_ord[['order_id', 'order_status', 'payment_method']].copy()
# Filter for cancelled orders
df_ques8 = df_ques8[df_ques8['order_status'] == 'cancelled']
# Inspect data types
df_ques8.info()

<class 'pandas.core.frame.DataFrame'>
Index: 59462 entries, 16 to 646926
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   order_id        59462 non-null  string  
 1   order_status    59462 non-null  category
 2   payment_method  59462 non-null  category
dtypes: category(2), string(1)
memory usage: 1.0 MB


In [115]:
# Print result
print(df_ques8['payment_method'].value_counts())
mode_payment = df_ques8['payment_method'].mode()[0]
print("\nThe most commonly used payment method is:", mode_payment)

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

The most commonly used payment method is: credit_card


#### -> Choose A

## Question 9: 
Among the four product sizes (*S*, *M*, *L*, *XL*), which size has the highest return rate, defined as the number of records in `returns` divided by the number of rows in `order_items` (joined with `products` on `product_id`)?

In [116]:
# Extract necessary columns
df_ques9_ret = df_ret[['return_id', 'order_id', 'product_id', 'return_quantity']].copy()
df_ques9_item = df_item[['order_id' ,'product_id', 'quantity']].copy()
df_support = df_pduct[['product_id', 'size']].copy()
# Inspect data types
df_ques9_ret.info()
df_ques9_item.info()
df_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   return_id        39939 non-null  string
 1   order_id         39939 non-null  string
 2   product_id       39939 non-null  string
 3   return_quantity  39939 non-null  int64 
dtypes: int64(1), string(3)
memory usage: 1.2 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   order_id    714669 non-null  string
 1   product_id  714669 non-null  string
 2   quantity    714669 non-null  int64 
dtypes: int64(1), string(2)
memory usage: 16.4 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   product_id  2

In [117]:
# Merge product size into the returns dataframe using product_id
df_ques9_ret = df_ques9_ret.merge(df_support, on='product_id', how='left')
# Merge product size into the order items dataframe using product_id
df_ques9_item = df_ques9_item.merge(df_support, on='product_id', how='left')
# Count total order items sold per size
sale = df_ques9_item.groupby('size', observed=True).size()
# Count total returns per size
returns = df_ques9_ret.groupby('size', observed=True).size()
# Print result
print(returns / sale)
print(f"\nThe product size with the highest return rate is: {(returns / sale).idxmax()}")

size
L     0.056250
M     0.055660
S     0.056515
XL    0.055200
dtype: float64

The product size with the highest return rate is: S


#### -> Choose A

## Question 10: 
In `payments.csv`, which installment plan has the highest average payment value per order?

In [118]:
df_ques10 = df_ord[['order_id', 'payment_value', 'installments']].copy()
# Inspect data types and missing values
df_ques10.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_id       646945 non-null  string 
 1   payment_value  646945 non-null  float64
 2   installments   646945 non-null  int64  
dtypes: float64(1), int64(1), string(1)
memory usage: 14.8 MB


In [119]:
# Calculate the average payment value per order for each installment plan.
average = df_ques10.groupby('installments')['payment_value'].mean()
# Print result
print(average)
print(f"\nThe installment plan with the highest average payment is: {average.idxmax()}")

installments
1     24113.274166
2       708.473729
3     24399.635486
6     24446.654403
12    24245.772694
Name: payment_value, dtype: float64

The installment plan with the highest average payment is: 6


#### -> Choose C